# Study 9: GP Parameter Inference via pyVBMC

Uses pyVBMC (Variational Bayesian Monte Carlo) to infer the posterior over GP parameters
`(length_scale, mu_0)` given participant prevalence ratings — replacing the coarse grid search
in `model-study9-param.ipynb` with principled Bayesian inference.

## What pyVBMC does
pyVBMC approximates the posterior `P(θ | ratings)` where `θ = (length_scale, mu_0)` using
a Gaussian mixture variational approximation. It is designed for expensive, noisy black-box
log-likelihoods — exactly our case, where each evaluation requires running MCMC.

## The objective
For each candidate `θ = (length_scale, mu_0)`, the log joint is:
```
log P(ratings, θ) = log P(ratings | θ) + log P(θ)
```
where `log P(ratings | θ)` is the Option 3 Monte Carlo log likelihood (logsumexp over
thinned MCMC samples), and `log P(θ)` is a broad log-normal prior on length_scale and
a Gaussian prior on mu_0.

## Parameter space
pyVBMC works in an **unconstrained** space. We transform:
- `length_scale > 0` → `log_ls = log(length_scale)` (unconstrained)
- `mu_0 ∈ ℝ` → already unconstrained

So the 2D unconstrained parameter vector is `φ = (log_ls, mu_0)`.

In [1]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"  # Metal incompatible with blackjax

import sys, csv, pickle as pkl
sys.path.insert(0, ".")

import numpy as np
import jax
import jax.numpy as jnp
import blackjax
import matplotlib.pyplot as plt
from scipy.special import logsumexp as scipy_logsumexp
from pyvbmc import VBMC

from model_jax import (
    make_log_density_fn_joint,
    fit_beta_mixtures_all_features,
    beta_mixture_log_likelihood,
)

print(f"backend: {jax.default_backend()}")

backend: cpu


In [2]:
with open('../features/set2_features_dataframe.pkl', 'rb') as f:
    df = pkl.load(f)

with open('../../data/study9.csv') as f:
    rows = list(csv.reader(f))
CSV_HEADER = rows[0]
data_rows  = rows[3:]

print(f"Features: {len(df)} total, {(df.split=='train').sum()} train, {(df.split=='test').sum()} test")
print(f"Participants: {len(data_rows)}")

Features: 60 total, 45 train, 15 test
Participants: 402


In [3]:
CSV_TO_FEATURE = {
    'diet_can_eat_spicy_1':  'can eat spicy food',
    'diet_breakfast_late_1': 'eat breakfast very late',
    'diet_five_meals_day_1': 'eat five meals a day',
    'diet_like_juice_pulp_1':'like juice with pulp',
    'diet_pepper_on_all_1':  'put pepper on all their foods',
    'pers_cry_easily_1':     'cry easily',
    'pers_collect_rocks_1':  'like to collect rocks',
    'pers_like_to_dance_1':  'like to dance',
    'pers_like_highfive_1':  'like to give high-fives',
    'pers_read_books_1':     'like to read books',
    'phys_can_roll_tongue_1':'can roll their tongue',
    'phys_can_snap_toes_1':  'can snap with their toes',
    'phys_can_wiggle_ears_1':'can wiggle their ears',
    'phys_cold_hands_feet_1':'have cold hands and feet',
    'phys_snore_sleep_1':    'snore when they sleep',
}
TEST_CSV_COLS = list(CSV_TO_FEATURE.keys())
feat_idx = df.set_index('feature')

train_df = df[df.split == 'train']
x_train  = jnp.array(train_df[['x_2d', 'y_2d']].values)
u_train  = jnp.zeros(len(train_df), dtype=jnp.int32)

test_feature_names = [CSV_TO_FEATURE[c] for c in TEST_CSV_COLS]
x_test = jnp.array(
    [feat_idx.loc[name, ['x_2d', 'y_2d']].values for name in test_feature_names]
)  # (J, 2)

col_indices = [CSV_HEADER.index(c) for c in TEST_CSV_COLS]
ratings = []
for row in data_rows:
    try:
        vals = [int(row[i]) / 100.0 for i in col_indices]
        ratings.append(vals)
    except (ValueError, IndexError):
        pass
responses = jnp.array(ratings)  # (N, J)
N, J = responses.shape

print(f"x_train: {x_train.shape}, x_test: {x_test.shape}, responses: {responses.shape}")

x_train: (45, 2), x_test: (15, 2), responses: (402, 15)


In [4]:
FIXED_PARAMS = {
    'output_scale': 1.5,
    'beta':         3.0,
}

# MCMC settings — kept lighter than grid search since pyVBMC calls this ~50-100 times
N_WARMUP  = 300
N_SAMPLES = 1000
STEP      = 5    # thinning: 999 → ~200 samples for Option 3

In [5]:
eval_count = [0]  # mutable counter for tracking evaluations

def log_joint(phi):
    """
    Log joint P(ratings, θ) for pyVBMC.

    phi: 1D array [log_ls, mu_0]  (unconstrained)
    Returns: scalar log joint
    """
    log_ls, mu_0 = float(phi[0]), float(phi[1])
    length_scale = np.exp(log_ls)

    eval_count[0] += 1
    print(f"  eval {eval_count[0]:3d}: ls={length_scale:.3f}  mu_0={mu_0:.3f}", end="  ")

    # --- log prior ---
    # length_scale: log-normal with median 0.5, broad sigma=1.5 in log space
    log_prior_ls  = -0.5 * ((log_ls - np.log(0.5)) / 1.5) ** 2
    # mu_0: Gaussian(0, 1)
    log_prior_mu0 = -0.5 * mu_0 ** 2
    log_prior = log_prior_ls + log_prior_mu0

    # --- log likelihood via MCMC + Option 3 ---
    params = {**FIXED_PARAMS, 'length_scale': length_scale, 'mu_0': mu_0}
    log_density_fn = make_log_density_fn_joint(u_train, x_train, x_test, params)

    init_position = {
        'training_coherences': jnp.zeros(x_train.shape[0]),
        'test_coherences':     jnp.zeros(J),
    }

    rng_key = jax.random.PRNGKey(0)
    rng_key, warmup_key = jax.random.split(rng_key)
    warmup = blackjax.window_adaptation(blackjax.nuts, log_density_fn)
    (state, nuts_params), _ = warmup.run(warmup_key, init_position, num_steps=N_WARMUP)

    nuts = blackjax.nuts(log_density_fn, **nuts_params)

    @jax.jit
    def one_step(state, key):
        return nuts.step(key, state)

    keys = jax.random.split(rng_key, N_SAMPLES)
    test_coherences_all = []
    for key in keys[:-1]:
        state, _ = one_step(state, key)
        test_coherences_all.append(np.array(state.position['test_coherences']))
    test_coherences_all = np.array(test_coherences_all)  # (S, J)

    # Option 3 log likelihood
    log_liks_s = []
    for s in range(0, test_coherences_all.shape[0], STEP):
        pz1_s    = jnp.array(jax.nn.sigmoid(test_coherences_all[s]))  # (J,)
        pz1_s_bc = jnp.tile(pz1_s, (N, 1))                            # (N, J)
        beta_params_s = fit_beta_mixtures_all_features(responses, pz1_s_bc)
        log_liks_s.append(beta_mixture_log_likelihood(responses, pz1_s, beta_params_s))

    log_lik = float(scipy_logsumexp(log_liks_s) - np.log(len(log_liks_s)))
    log_joint_val = log_lik + log_prior
    print(f"log_lik={log_lik:.1f}  log_joint={log_joint_val:.1f}")
    return log_joint_val

## pyVBMC setup

pyVBMC needs:
- `x0`: starting point in unconstrained space `[log_ls, mu_0]`
- `lb`, `ub`: hard bounds (log_ls in ~[-3, 3] → ls in [0.05, 20]; mu_0 in [-3, 3])
- `plb`, `pub`: plausible bounds (where most of the posterior mass is expected)

Starting from `(ls=0.4, mu_0=0.0)` which was near the grid search best.

In [6]:
# Starting point: ls=0.4, mu_0=0.0
x0  = np.array([np.log(0.4), 0.0])

# Hard bounds
lb  = np.array([np.log(0.05), -4.0])  # ls >= 0.05, mu_0 >= -4
ub  = np.array([np.log(20.0),  4.0])  # ls <= 20,   mu_0 <= 4

# Plausible bounds (where we expect most posterior mass based on grid search)
plb = np.array([np.log(0.1), -2.0])
pub = np.array([np.log(5.0),  2.0])

print(f"Starting point: log_ls={x0[0]:.2f} (ls={np.exp(x0[0]):.2f}), mu_0={x0[1]:.2f}")
print(f"Bounds: ls in [{np.exp(lb[0]):.2f}, {np.exp(ub[0]):.2f}], mu_0 in [{lb[1]}, {ub[1]}]")
print()

vbmc = VBMC(log_joint, x0, lb, ub, plb, pub)
vbmc_result, vbmc_stats = vbmc.optimize()

Starting point: log_ls=-0.92 (ls=0.40), mu_0=0.00
Bounds: ls in [0.05, 20.00], mu_0 in [-4.0, 4.0]

Reshaping x0 to row vector.
Reshaping lower bounds to (1, 2).
Reshaping upper bounds to (1, 2).
Reshaping plausible lower bounds to (1, 2).
Reshaping plausible upper bounds to (1, 2).
Beginning variational optimization assuming EXACT observations of the log-joint.
 Iteration  f-count    Mean[ELBO]    Std[ELBO]    sKL-iter[q]   K[q]  Convergence  Action
  eval   1: ls=0.400  mu_0=0.000  

/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=c

log_lik=134591.7  log_joint=134591.7
  eval   2: ls=0.112  mu_0=-1.872  

/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=c

log_lik=10267660849.9  log_joint=10267660847.6
  eval   3: ls=0.254  mu_0=-0.650  

...

/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=c

log_lik=201331157.8  log_joint=201331156.2
  eval  10: ls=0.483  mu_0=-0.482  

/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=copy)
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/scipy/_lib/array_api_compat/numpy/_aliases.py:106: RuntimeWarning: overflow encountered in cast
  return x.astype(dtype=dtype, copy=c

log_lik=2415922839.4  log_joint=2415922839.2


ValueError: setting an array element with a sequence.

In [ ]:
# Extract posterior samples from the variational approximation
phi_samples, log_weights = vbmc_result.vp.sample(int(1e4))
# phi_samples: (10000, 2) in unconstrained space [log_ls, mu_0]

ls_samples  = np.exp(phi_samples[:, 0])  # back-transform to length_scale
mu0_samples = phi_samples[:, 1]

print(f"Posterior mean:   ls={ls_samples.mean():.3f}  mu_0={mu0_samples.mean():.3f}")
print(f"Posterior median: ls={np.median(ls_samples):.3f}  mu_0={np.median(mu0_samples):.3f}")
print(f"Posterior std:    ls={ls_samples.std():.3f}  mu_0={mu0_samples.std():.3f}")
print(f"95% CI ls:  [{np.percentile(ls_samples, 2.5):.3f}, {np.percentile(ls_samples, 97.5):.3f}]")
print(f"95% CI mu_0: [{np.percentile(mu0_samples, 2.5):.3f}, {np.percentile(mu0_samples, 97.5):.3f}]")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Marginal: length_scale
axes[0].hist(ls_samples, bins=50, density=True, color='steelblue', alpha=0.7)
axes[0].axvline(np.median(ls_samples), color='tomato', lw=2, label=f'median={np.median(ls_samples):.2f}')
axes[0].set_xlabel('length_scale')
axes[0].set_ylabel('density')
axes[0].set_title('Posterior: length_scale')
axes[0].legend()

# Marginal: mu_0
axes[1].hist(mu0_samples, bins=50, density=True, color='seagreen', alpha=0.7)
axes[1].axvline(np.median(mu0_samples), color='tomato', lw=2, label=f'median={np.median(mu0_samples):.2f}')
axes[1].set_xlabel('mu_0')
axes[1].set_ylabel('density')
axes[1].set_title('Posterior: mu_0')
axes[1].legend()

# Joint posterior
axes[2].scatter(ls_samples[::10], mu0_samples[::10], alpha=0.1, s=2, color='purple')
axes[2].set_xlabel('length_scale')
axes[2].set_ylabel('mu_0')
axes[2].set_title('Joint posterior (thinned)')

plt.tight_layout()
plt.show()

In [ ]:
# Forward pass at posterior mean params
ls_mean  = float(ls_samples.mean())
mu0_mean = float(mu0_samples.mean())
print(f"Running forward pass at posterior mean: ls={ls_mean:.3f}, mu_0={mu0_mean:.3f}")

params_best = {**FIXED_PARAMS, 'length_scale': ls_mean, 'mu_0': mu0_mean}
log_density_fn = make_log_density_fn_joint(u_train, x_train, x_test, params_best)

init_position = {
    'training_coherences': jnp.zeros(x_train.shape[0]),
    'test_coherences':     jnp.zeros(J),
}

rng_key = jax.random.PRNGKey(0)
rng_key, warmup_key = jax.random.split(rng_key)
warmup = blackjax.window_adaptation(blackjax.nuts, log_density_fn)
(state, nuts_params), _ = warmup.run(warmup_key, init_position, num_steps=500)

nuts = blackjax.nuts(log_density_fn, **nuts_params)

@jax.jit
def one_step(state, key):
    return nuts.step(key, state)

keys = jax.random.split(rng_key, 2000)
test_coherences_all = []
for key in keys[:-1]:
    state, _ = one_step(state, key)
    test_coherences_all.append(np.array(state.position['test_coherences']))
test_coherences_all = np.array(test_coherences_all)  # (S, J)

pz1_test = np.array([
    float(np.mean(jax.nn.sigmoid(jnp.array(test_coherences_all[:, j]))))
    for j in range(J)
])

# Option 3 log likelihood + predicted prevalence
pred_prevalences_s = []
log_liks_s = []
for s in range(0, test_coherences_all.shape[0], 10):
    pz1_s    = jnp.array(jax.nn.sigmoid(test_coherences_all[s]))
    pz1_s_bc = jnp.tile(pz1_s, (N, 1))
    beta_params_s = fit_beta_mixtures_all_features(responses, pz1_s_bc)
    a_kl  = np.array(beta_params_s[:, 0]); b_kl  = np.array(beta_params_s[:, 1])
    a_nkl = np.array(beta_params_s[:, 2]); b_nkl = np.array(beta_params_s[:, 3])
    pred_prevalences_s.append(
        np.array(pz1_s) * (a_kl/(a_kl+b_kl)) + (1-np.array(pz1_s)) * (a_nkl/(a_nkl+b_nkl))
    )
    log_liks_s.append(beta_mixture_log_likelihood(responses, pz1_s, beta_params_s))

pred_prevalence = np.mean(pred_prevalences_s, axis=0)
log_lik_best = float(scipy_logsumexp(log_liks_s) - np.log(len(log_liks_s)))

empirical_mean = np.array(responses.mean(axis=0))
corr = np.corrcoef(empirical_mean, pred_prevalence)[0, 1]
print(f"log_lik={log_lik_best:.1f}  Pearson r={corr:.3f} (reference only)")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(empirical_mean, pred_prevalence, zorder=3)
for j, name in enumerate(test_feature_names):
    ax.annotate(name, (empirical_mean[j], pred_prevalence[j]),
                fontsize=7, xytext=(4, 2), textcoords='offset points')
lims = [0, 1]
ax.plot(lims, lims, '--', color='gray', alpha=0.5)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Empirical mean prevalence')
ax.set_ylabel("Model E[p' | u]")
ax.set_title(f'Forward pass at posterior mean params\nls={ls_mean:.3f}, mu_0={mu0_mean:.3f}  log_lik={log_lik_best:.1f}')
plt.tight_layout()
plt.show()